# ChessLab Explore Chess.com Data Analytics

## Goals:
- Load some data from previous months
- Inspect what Chess.com returns before deciding what analytics to build
- Plot some data to see what might be useful for final output

## 1. Imports and settings

Start with one recent month. You can increase the number later.

In [1]:
from pathlib import Path
import re
import sqlite3

import pandas as pd
import plotly.express as px
from IPython.display import display

from chesslab.api.chesscom import get_game_archives, download_monthly_games
from chesslab.chess.parser import parse_games, get_game_info
from chesslab.storage.database import (
    create_connection,
    create_games_table,
    save_game,
)

# ensure directories are set up correctly and create a database in data/ 
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
DATABASE_PATH = DATA_DIR / "chesslab.db"

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Database path: {DATABASE_PATH.resolve()}")

Project root: /Users/timmymooney
Database path: /Users/timmymooney/data/chesslab.db


## 2. Get Game Data Archives

In [2]:
# define username and number of months to fetch
USERNAME = "tjmoonz98"
MONTHS_TO_FETCH = 4

archives = get_game_archives(USERNAME)
print(f"Monthly archives available for {USERNAME}: {len(archives)}")

selected_archives = archives[-MONTHS_TO_FETCH:]
print("Archives selected:")
for archive_url in selected_archives:
    print(archive_url)

Monthly archives available for tjmoonz98: 8
Archives selected:
https://api.chess.com/pub/player/tjmoonz98/games/2026/08
https://api.chess.com/pub/player/tjmoonz98/games/2026/09
https://api.chess.com/pub/player/tjmoonz98/games/2026/10
https://api.chess.com/pub/player/tjmoonz98/games/2026/11


## 3. Download the raw Chess.com Data

In [3]:
monthly_pgns = []

for archive_url in selected_archives:
    match = re.search(r"/games/(\d{4})/(\d{2})$", archive_url)
    if not match:
        print(f"Skipping archive with an unexpected URL format: {archive_url}")
        continue

    year, month = map(int, match.groups())
    month_pgn = download_monthly_games(USERNAME, year, month)

    if month_pgn.strip():
        monthly_pgns.append(month_pgn)
        print(f"Downloaded {year}-{month:02d}: {len(month_pgn):,} characters")
    else:
        print(f"No PGN data for {year}-{month:02d}")

combined_pgn = "\n\n".join(monthly_pgns)
print(f"Total downloaded PGN characters: {len(combined_pgn):,}")

Downloaded 2026-08: 195,082 characters
Downloaded 2026-09: 187,579 characters
Downloaded 2026-10: 144,701 characters
Downloaded 2026-11: 41,522 characters
Total downloaded PGN characters: 568,890


In [4]:
games = parse_games(combined_pgn)
print(f"Games parsed: {len(games)}")

if not games:
    raise ValueError(
        "No games were parsed. Check USERNAME, the selected archives, and the downloaded PGN."
    )

games_df = pd.DataFrame(
    [get_game_info(game, USERNAME) for game in games]
)

display(games_df)
print(f"Rows: {len(games_df):,} | Columns: {len(games_df.columns)}")

Games parsed: 261


,username,white,black,result,event,date,eco,white_elo,black_elo,time_control,termination,link
0,tjmoonz98,tjmoonz98,Rejielonile,1-0,Live Chess,2026.08.31,D02,463,469,900+10,tjmoonz98 won by resignation,https://www.chess.com/game/live/173811050396
1,tjmoonz98,banban233,tjmoonz98,0-1,Let's Play!,2026.08.25,D00,158,609,1/259200,tjmoonz98 won by checkmate,https://www.chess.com/game/daily/1019309082
2,tjmoonz98,tjmoonz98,dash27darshan,1-0,Live Chess,2026.08.29,C42,454,456,900+10,tjmoonz98 won by resignation,https://www.chess.com/game/live/173719533368
3,tjmoonz98,tjmoonz98,banban233,1-0,Live Chess,2026.08.29,B20,446,214,600,tjmoonz98 won by checkmate,https://www.chess.com/game/live/173693139946
4,tjmoonz98,tjmoonz98,ChessUsernameLetMePlay,1-0,Let's Play!,2026.08.28,C41,580,574,1/86400,tjmoonz98 won by resignation,https://www.chess.com/game/daily/1020776964
...,...,...,...,...,...,...,...,...,...,...,...,...
256,tjmoonz98,PSausH,tjmoonz98,0-1,Let's Play!,2026.11.01,A00,820,827,1/259200,tjmoonz98 won by resignation,https://www.chess.com/game/daily/1054710048
257,tjmoonz98,rider540,tjmoonz98,0-1,Live Chess,2026.11.02,C20,500,523,600,tjmoonz98 won by resignation,https://www.chess.com/game/live/186418578488
258,tjmoonz98,tjmoonz98,akhdan8934,1-0,Live Chess,2026.11.02,A00,515,492,600,tjmoonz98 won by checkmate,https://www.chess.com/game/live/186416823066
259,tjmoonz98,tjmoonz98,FMF0301,0-1,Live Chess,2026.11.01,C20,507,522,600,FMF0301 won by checkmate,https://www.chess.com/game/live/186364941360


Rows: 261 | Columns: 12


## 4. Prepare Data

In [5]:
# identify which colour i played
my_white = games_df["white"].str.lower().eq(USERNAME.lower())
my_black = games_df["black"].str.lower().eq(USERNAME.lower())

games_df["my_colour"] = None
games_df.loc[my_white, "my_colour"] = "White"
games_df.loc[my_black, "my_colour"] = "Black"

# identify my result
def get_your_result(row):
    if row["my_colour"] not in ["White", "Black"]:
        return "Unknown"
    if row["result"] == "1/2-1/2":
        return "Draw"
    elif row["result"] == "1-0":
        return "Win" if row["my_colour"] == "White" else "Loss"
    elif row["result"] == "0-1":
        return "Win" if row["my_colour"] == "Black" else "Loss"
    return "Unknown"

games_df["my_result"] = games_df.apply(get_your_result, axis=1)

display(games_df[["white", "black", "my_colour", "my_result"]].head())


,white,black,my_colour,my_result
0,tjmoonz98,Rejielonile,White,Win
1,banban233,tjmoonz98,Black,Win
2,tjmoonz98,dash27darshan,White,Win
3,tjmoonz98,banban233,White,Win
4,tjmoonz98,ChessUsernameLetMePlay,White,Win


In [6]:
# mutate new column for the chess types
time_control_mapping = {
    "1800": "Rapid",
    "900+10": "Rapid",
    "600": "Rapid",
    "300": "Blitz",
    "1/86400": "Daily",
    "1/259200": "Daily",
    "1/864000": "Daily",
    "-": "Other",
}

games_df["chess_type"] = games_df["time_control"].map(time_control_mapping)

games_df.value_counts('chess_type')

chess_type
Rapid    193
Daily     66
Other      1
Blitz      1
Name: count, dtype: int64

## 5. Simple Win/Loss/Draw Plot

In [7]:
# count results by chess colour
colour_counts = (
    games_df.groupby(["my_colour", "my_result"])
    .size()
    .reset_index(name="count")
)

# create bar chart comparing white and black
fig = px.bar(
    colour_counts,
    x="my_result",
    y="count",
    color="my_colour",
    barmode="group",
    color_discrete_map={
        "White": "#C9C9C9",
        "Black": "#404040"
    },
    category_orders={
        "my_colour": ["White", "Black"],
        "my_result": ["Win", "Draw", "Loss"]
    },
    title="My Chess Results: White vs Black",
    labels={
        "my_result": "Result",
        "count": "Number of Games",
        "my_colour": "Chess Colour"
    },
    text="count"
)

fig.show()

In [8]:
# calculate win percentage for each colour
win_percentage = (
    games_df.groupby("my_colour")["my_result"]
    .apply(lambda x: (x == "Win").mean() * 100)
    .reset_index(name="win_percentage")
)

# plot win percentage by colour
fig = px.bar(
    win_percentage,
    x="my_colour",
    y="win_percentage",
    color="my_colour",
    color_discrete_map={
        "White": "#C9C9C9",
        "Black": "#404040"
    },
        category_orders={
        "my_colour": ["White", "Black"]
    },
    title="My Win Percentage: White vs Black",
    labels={
        "my_colour": "Chess Colour",
        "win_percentage": "Win Percentage (%)"
    },
    text_auto=".1f"
)

fig.update_yaxes(range=[0, 100])
fig.show()

## 6. Rating over time

The PGN ratings are snapshots at game time. We can plot them, but they do not directly tell us the rating gained or lost in each game. Different chess types have separate rating pools.


In [15]:
# prepare date and my rating from the existing pgn columns
games_df["date"] = pd.to_datetime(games_df["date"], format="%Y.%m.%d", errors="coerce")
games_df["my_rating"] = pd.to_numeric(
    games_df["white_elo"].where(games_df["my_colour"] == "White", games_df["black_elo"]),
    errors="coerce"
)

# keep games with a date, rating and recognised chess type
rating_df = games_df.dropna(subset=["date", "my_rating", "chess_type"])
rating_df = rating_df[rating_df["chess_type"].isin(["Rapid", "Blitz", "Daily"])]
rating_df = rating_df.sort_values("date")

fig = px.line(
    rating_df,
    x="date",
    y="my_rating",
    color="chess_type",
    markers=True,
    title="My Chess Rating Over Time",
    labels={"date": "Date", "my_rating": "Rating", "chess_type": "Chess Type"},
    color_discrete_map={"Rapid": "#13E01D", "Blitz": "#FFF700", "Daily": "#FFCC00"},
)
fig.show()

## 7. Group players as me or opponent

Keep the original `white` and `black` columns, but create simpler labels for plotting: `tjmoonz98` or `Opponent`.

In [10]:
# label each player as me or opponent
games_df["white_player"] = games_df["white"].apply(
    lambda name: USERNAME if str(name).lower() == USERNAME.lower() else "Opponent"
)
games_df["black_player"] = games_df["black"].apply(
    lambda name: USERNAME if str(name).lower() == USERNAME.lower() else "Opponent"
)

display(games_df[["white_player", "black_player", "my_colour", "my_result"]].head())

,white_player,black_player,my_colour,my_result
0,tjmoonz98,Opponent,White,Win
1,Opponent,tjmoonz98,Black,Win
2,tjmoonz98,Opponent,White,Win
3,tjmoonz98,Opponent,White,Win
4,tjmoonz98,Opponent,White,Win


## 8. How did my games end, and who won?

Use the `termination` description to identify how the game ended, then combine it with `my_result` to distinguish my wins from opponent wins. Draws are shown separately. The `Other` category catches descriptions we cannot classify reliably.

In [11]:
# classify how the game ended
def get_end_reason(termination):
    text = str(termination).lower()
    if "checkmate" in text:
        return "Checkmate"
    if "resign" in text:
        return "Resignation"
    if "time" in text:
        return "Timeout"
    if "abandon" in text:
        return "Abandonment"
    return "Other"

games_df["end_reason"] = games_df["termination"].apply(get_end_reason)

# combine the winner and the ending reason
def get_game_ending(row):
    if row["my_result"] == "Draw":
        return "Draw"
    if row["my_result"] == "Win":
        return f"{USERNAME} wins by {row['end_reason'].lower()}"
    if row["my_result"] == "Loss":
        return f"Opponent wins by {row['end_reason'].lower()}"
    return "Unknown"

games_df["game_ending"] = games_df.apply(get_game_ending, axis=1)

display(games_df["game_ending"].value_counts())

game_ending
tjmoonz98 wins by checkmate      61
Opponent wins by resignation     53
tjmoonz98 wins by resignation    52
Opponent wins by checkmate       44
tjmoonz98 wins by timeout        20
tjmoonz98 wins by abandonment    16
Draw                             12
Opponent wins by timeout          2
Opponent wins by abandonment      1
Name: count, dtype: int64

## 9. Plot game endings

First plot the detailed endings (such as `tjmoonz98 wins by checkmate` versus `Opponent wins by checkmate`), then compare wins and losses by ending reason.

In [12]:
# count each game ending
ending_counts = games_df["game_ending"].value_counts().reset_index()
ending_counts.columns = ["game_ending", "count"]

fig = px.bar(
    ending_counts,
    x="count",
    y="game_ending",
    orientation="h",
    title="How My Games Ended",
    labels={"game_ending": "Game Ending", "count": "Games"},
    text="count",
    color_discrete_sequence=["#606060"],
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

# compare my wins and losses for each ending reason
ending_by_result = (
    games_df[games_df["my_result"].isin(["Win", "Loss"])]
    .groupby(["end_reason", "my_result"])
    .size()
    .reset_index(name="count")
)

fig = px.bar(
    ending_by_result,
    x="end_reason",
    y="count",
    color="my_result",
    barmode="group",
    color_discrete_map={"Win": "#7A9B83", "Loss": "#A65E5E"},
    title="Game Endings: My Wins vs Opponent Wins",
    labels={"end_reason": "Ending Reason", "count": "Games", "my_result": "Result"},
    text="count",
)
fig.show()

## What We've Learned

We now have results by colour, win percentage by colour, rating history by chess type and game-ending charts that distinguish your wins from opponent wins.